In [1]:
###### nn.Module
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(784, 256)   # MNIST 이미지는 28x28여서
        self.fc2=nn.Linear(256, 128)
        self.fc3=nn.Linear(128, 10)
        self.relu=nn.ReLU()

    def forward(self, x):
        x=x.view(-1, 784)
        x=self.relu(self.fc1(x))
        x=self.relu(self.fc2(x))
        x=self.fc3(x)
        return x

In [3]:
### 데이터 변환에 필요한 단계
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

### 데이터 다운로드
train_dataset=datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset=datasets.MNIST('./data', train=False, transform=transform)

### 데이터들 64개씩 꺼내주는 기계
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)  

In [4]:
### 모델, 손실함수, 옵티마이저 선정
model=SimpleNet() 
criterion=nn.CrossEntropyLoss() 
optimizer=optim.Adam(model.parameters(), lr=0.001)

In [5]:
### 학습 함수
def train(model, loader, criterion, optimizer, epoch):
    model.train()
    total_loss=0
    
    for batch_idx, (data, target) in enumerate(loader):
        optimizer.zero_grad()
        output=model(data)

        loss=criterion(output, target)
        loss.backward()

        optimizer.step()
        total_loss+=loss.item()
        
    print(f"epoch{epoch}: loss={total_loss/len(loader):.4f}")

In [6]:
def evaluate(model, loader):
    model.eval()
    correct=0
    
    with torch.no_grad():
        for data, target in loader:
            output=model(data)
            pred=output.argmax(dim=1)
            correct+=pred.eq(target).sum().item()
            
    acc=100. * correct / len(loader.dataset)
    
    print(f"accuracy:{acc:.2f}%")
    
    return acc

In [7]:
for epoch in range(1,6):
    train(model, train_loader, criterion, optimizer, epoch)
    evaluate(model, test_loader)

epoch1: loss=0.2286
accuracy:96.28%
epoch2: loss=0.0919
accuracy:96.63%
epoch3: loss=0.0650
accuracy:97.22%
epoch4: loss=0.0502
accuracy:97.40%
epoch5: loss=0.0412
accuracy:97.71%
